# Python Casting
> 📘 **Python Mastery** · Module 01 — Python Basics · Lesson 4/8

Python is dynamically typed — a variable can hold any type, and the type can change at runtime. But sometimes you need to be explicit. **Casting** (also called *type conversion*) lets you transform values from one type to another, such as turning the string `"42"` into the integer `42`. This lesson covers the built-in cast functions, their edge cases, and the patterns you will use every day in real Python code.

## 🎯 Learning Objectives

- **Explain** the difference between Python's automatic type inference and explicit type casting.
- **Apply** `int()`, `float()`, and `str()` to convert between common types safely.
- **Distinguish** truncation (what `int()` does to floats) from rounding (what you might expect).
- **Handle** invalid cast attempts with `try/except` and the `.isdigit()` guard pattern.
- **Predict** the result of `bool()` for any Python value using the falsy/truthy rules.
- **Apply** casting in real-world contexts: user input, CSV cleaning, API responses, and data pipelines.

## 1. What Is Casting?

**Casting** (or *explicit type conversion*) means turning a value from one type into another by calling a type function. You have already seen this in other languages — Java's `(int)`, C++'s `static_cast`, or JavaScript's `Number()`.

Python does not require casting for most operations (it uses *duck typing*), but there are three situations where you must be explicit:

1. **Math on strings** — you cannot add `str + int`; you must cast the string first.
2. **Concatenating non-strings** — `print(3 + " apples")` raises a `TypeError`.
3. **APIs and file formats** — external systems expect a specific type (JSON wants strings for large numbers, databases want typed values, etc.).

Python also performs **implicit casting** automatically: `5 + 2.0` becomes `7.0` without you asking. Python promotes the `int` to `float` so the operation succeeds. You never need to cast for basic arithmetic between compatible numeric types.

In [1]:
x = "5"      # string
y = int(x)   # explicit: str -> int
print(y + 3) # 8  (math works now)
print(type(x), type(y))

8
<class 'str'> <class 'int'>


## 2. The Three Cast Functions

Python has three primary built-in cast functions. They accept one argument and return the converted value.

| Function | Accepts | Returns | Notes |
| --- | --- | --- | --- |
| `int(x)` | `float`, `str`, `bool` | `int` | Truncates floats; parses integer strings |
| `float(x)` | `int`, `str`, `bool` | `float` | Adds `.0` to integers; parses decimal strings |
| `str(x)` | `int`, `float`, `bool`, `None`, any object | `str` | Human-readable representation |

In [2]:
# int() conversions
print(int(3.9))        # 3  -> truncates, does NOT round
print(int("42"))       # 42 -> parses a string
print(int(True))      # 1  -> bool is a subclass of int

# float() conversions
print(float(7))        # 7.0 -> adds decimal point
print(float("3.14"))   # 3.14
print(float("1e3"))    # 1000.0 -> scientific notation works

# str() conversions
print(str(100))        # "100"
print(str(3.14))       # "3.14"
print(str(True))       # "True"
print(str(None))       # "None"

3
42
1
7.0
3.14
1000.0
100
3.14
True
None


## 3. `int()` Deep Dive

The `int()` function converts values to integers. Its behaviour varies depending on the input type:

### From `float`: Truncation, Not Rounding

`int(3.9)` returns `3`, not `4`. Python discards the fractional part entirely — this is **truncation toward zero**. For rounding, use `round(3.9)` or `math.floor(3.9)`.

### From `str`: Base Conversion

When given a string, `int()` can accept an optional second argument specifying the number's base:

- `int("1010", 2)` → `10` (binary)
- `int("FF", 16)` → `255` (hexadecimal)
- `int("42", 10)` → `42` (decimal, the default)
- `int("0xFF", 16)` → `255` (0x prefix also works with base 16)

### Invalid Strings Raise `ValueError`

If the string does not represent a valid integer in the given base, Python raises a `ValueError`. This is why you should always guard `int()` calls with `try/except` (covered in Section 8).

In [3]:
# Truncation vs rounding
print(int(7.9))       # 7  -> drops fractional part
print(int(-2.3))      # -2 -> truncates toward zero
print(round(7.9))     # 8  -> actual rounding

# Base conversion
print(int("1010", 2))     # 10  -> binary
print(int("0xFF", 16))    # 255 -> hexadecimal
print(int("A", 16))       # 10  -> single hex digit

# ValueError examples (uncomment to see errors)
# int("3.14")      # ValueError: invalid literal for int()
# int("hello")     # ValueError: invalid literal for int()
# int("")          # ValueError: invalid literal for int()

7
-2
8
10
255
10


## 4. `float()` Deep Dive

The `float()` function converts values to Python's 64-bit IEEE 754 floating-point numbers.

### Key Behaviors

**Scientific notation:** `float("3.5e2")` returns `350.0`. The `e` (or `E`) exponent notation is supported in string input.

**Precision limit:** Python floats have approximately **15-17 decimal digits of precision**. This is a hardware limitation — adding many small floats can accumulate tiny rounding errors. For financial calculations requiring exact decimal arithmetic, use the `decimal` module:

```python
from decimal import Decimal
price = Decimal("19.99")  # exact, not a float
```

### Binary Representation Warning

Many decimal values cannot be represented exactly in binary floating point. `0.1 + 0.2` is famously `0.30000000000000004` in Python. This is not a bug — it is how binary floats work on all computers. For display purposes, use `round()` or f-string formatting.

In [4]:
# Scientific notation in strings
print(float("3.5e2"))     # 350.0
print(float("1.23e-4"))   # 0.000123
print(float("1E6"))       # 1000000.0

# Precision demonstration
big_num = 1.234567890123456789
print(big_num)            # loses precision after ~17 digits

# The classic floating-point gotcha
result = 0.1 + 0.2
print(result)             # 0.30000000000000004
print(result == 0.3)       # False!
print(round(result, 2))    # 0.3  -> round for display

350.0
0.000123
1000000.0
1.2345678901234567
0.30000000000000004
False
0.3


## 5. `str()` — Not Just for Numbers

`str()` converts **any** Python object into its string representation. It is the universal "make it text" function.

### Conversions

`str()` handles integers, floats, booleans, `None`, lists, tuples, dictionaries, and even custom objects (by calling their `__str__` method).

### `repr()` vs `str()`

Python has two similar functions for turning objects into strings:

- **`str()`** — human-readable, omits quotes around strings, hides technical details.
- **`repr()`** — unambiguous, includes quotes, shows the *representation* a Python developer would write.

For simple types like `int` and `float`, both behave identically. The difference matters for strings (quotes) and debugging.

In [5]:
# Conversions to string
print(str(42))              # "42"
print(str(3.14))            # "3.14"
print(str(True))           # "True"
print(str([1, 2, 3]))       # "[1, 2, 3]"
print(str(None))           # "None"

# str() vs repr() — the difference is visible for strings
text = "hello"
print(str(text))           # hello      -> no quotes
print(repr(text))          # 'hello'    -> quotes shown

# repr() also escapes special characters
path = "C:\\Users\\Fahim"
print(repr(path))          # 'C:\\Users\\Fahim'

42
3.14
True
[1, 2, 3]
None
hello
'hello'
'C:\\Users\\Fahim'


## 6. Boolean Casting — Where Types Collide

The `bool()` function is the bridge between types and logic. It converts any value to `True` or `False` using Python's truthiness rules.

### Truthy vs Falsy

Every Python value is either **truthy** (evaluates to `True`) or **falsy** (evaluates to `False` when passed to `bool()`). There is a fixed, short list of falsy values:

| Falsy Value | Notes |
| --- | --- |
| `False` | the boolean itself |
| `None` | "no value" |
| `0`, `0.0`, `0j` | all zero types |
| `""` | empty string |
| `[]`, `()`, `{}`, `set()` | empty containers |
| `range(0)` | an empty range |

Everything else is truthy — including the string `"False"` and the number `-1`.

In [6]:
# Classic falsy values
print(bool(0))        # False
print(bool(0.0))      # False
print(bool(""))        # False  (empty string)
print(bool(None))      # False
print(bool([]))        # False  (empty list)

# Truthy values — things that look "false" but are not
print(bool(-1))        # True   -> non-zero is truthy
print(bool("False"))   # True   -> non-empty string is truthy
print(bool("0"))       # True   -> the character 0 is not zero
print(bool([0]))       # True   -> a list with one item
print(bool(" "))       # True   -> a space is content

False
False
False
False
False
True
True
True
True
True


## 7. Common Patterns

Casting appears constantly in real Python code. Here are the three situations you will encounter most often.

### Pattern 1: User Input

`input()` always returns a string. Any numeric computation or validation requires casting.

### Pattern 2: CSV Data Cleaning

Raw CSV data is all strings. Before calculating averages, totals, or performing comparisons, you must cast numeric columns.

### Pattern 3: API Response Parsing

JSON responses from REST APIs arrive as strings over HTTP. Python's `json.loads()` returns native types, but individual fields often need casting — especially when dealing with string-encoded timestamps, prices, or IDs.

In [7]:
# Pattern 1: User input conversion
age_str = "25"                          # simulating input()
age = int(age_str)                      # str -> int
print(f"In 10 years you will be {age + 10}")

# Pattern 2: CSV data cleaning (simulated)
csv_row = ["Alice", "28", "165.5", "False"]
name = csv_row[0]                       # already a string
age = int(csv_row[1])                   # "28" -> 28
height = float(csv_row[2])              # "165.5" -> 165.5
is_student = csv_row[3] == "True"       # careful with bool strings!
print(f"{name}: age={age}, height={height}, student={is_student}")

# Pattern 3: API response parsing
api_response = {
    "id": "12345",
    "price": "29.99",
    "in_stock": "true"
}
product_id = int(api_response["id"])
price = float(api_response["price"])
in_stock = api_response["in_stock"].lower() == "true"
print(f"Product #{product_id}: ${price} (available={in_stock})")

In 10 years you will be 35
Alice: age=28, height=165.5, student=False
Product #12345: $29.99 (available=True)


## 8. Handling Bad Casts

When casting fails, Python raises a `ValueError`. Two strategies handle this:

### Strategy 1: `try/except`

Wrap the potentially failing cast in a `try` block and catch `ValueError`. This is the most robust approach — it handles all invalid inputs in one place. (Error handling is covered in detail in a later module; this is a preview of the pattern.)

### Strategy 2: Guard with `.isdigit()` or `.isnumeric()`

For strings, you can check whether the string is safe to cast before attempting the conversion. This only works for positive integers; negative numbers and decimals require regex or `try/except`.

In [8]:
# Strategy 1: try/except — the robust approach
def safe_int(value):
    try:
        return int(value)
    except (ValueError, TypeError):
        return None          # or a default like 0

test_values = ["42", "3.14", "hello", "-7", ""]
for v in test_values:
    result = safe_int(v)
    print(f"int('{v}') -> {result}")

# Strategy 2: .isdigit() guard — for positive integers only
user_input = "123"
if user_input.isdigit():
    number = int(user_input)
    print(f"Valid input: {number}")
else:
    print(f"Invalid input: '{user_input}' is not a positive integer")

# .isdigit() fails for negative numbers
print("-42".isdigit())      # False  -> minus sign is not a digit
print("3.14".isdigit())     # False  -> decimal point is not a digit
print("42".isdigit())       # True   -> only positive integers pass

int('42') -> 42
int('3.14') -> None
int('hello') -> None
int('-7') -> -7
int('') -> None
Valid input: 123
False
False
True


## ⚠️ Common Mistakes & Gotchas

| Mistake | Problem | Fix |
| --- | --- | --- |
| `int(3.9)` rounding to `4` | `int()` **truncates** toward zero, it never rounds | Use `round(3.9)` for rounding, or `math.floor()` / `math.ceil()` |
| `float("1,000")` failing | The comma is not a valid decimal separator in Python | Strip the comma first: `float("1,000".replace(",", ""))` |
| `int("3.14")` raising `ValueError` | Python cannot directly convert a float string to int | Cast in two steps: `int(float("3.14"))` |
| `bool("False")` returning `True` | Any non-empty string is truthy, even `"False"` | Compare explicitly: `s == "True"` or `s.lower() == "true"` |
| `int(3+2j)` or `int(2j)` failing | Complex numbers cannot be cast to int or float directly | Access the real part: `int((3+2j).real)` |
| `int("  42  ")` failing | Leading and trailing spaces cause ValueError | Strip first: `int("  42  ".strip())` |

In [9]:
# Gotcha 1: int() truncates, not rounds
print(int(7.8))        # 7, not 8
print(int(-1.2))       # -1, not -2

# Gotcha 2: comma is not a decimal separator
# float("1,000")      # ValueError
print(float("1000"))           # works: 1000.0
print(float("1,000".replace(",", "")))  # strip the comma first

# Gotcha 3: float-string to int needs two steps
# int("3.14")          # ValueError
print(int(float("3.14")))      # 3  -> int(float()) works

# Gotcha 4: bool("False") is True!
print(bool("False"))           # True — the STRING "False", not the bool
print(bool(False))             # False — the actual bool

# Gotcha 5: complex cannot cast to int
# int(3+2j)            # TypeError
print((3+2j).real)             # 3.0  -> access parts separately

# Gotcha 6: whitespace in strings
# int("  42  ")        # ValueError
print(int("  42  ".strip()))   # 42  -> strip() removes whitespace

7
-1
1000.0
1000.0
3
True
False
3.0
42


## 💡 Best Practices & Pro Tips

- **Validate before casting.** If user input or external data might be malformed, check with `.isdigit()` or wrap in `try/except` rather than letting exceptions propagate.
- **Remember truncation.** `int(3.7)` is `3`, not `4`. If you need rounding, call `round()` first.
- **Strip strings.** Always `.strip()` user input before casting — unexpected whitespace is a common source of silent `ValueError` failures.
- **Use `float()` for prices and measurements.** Integers cannot represent cents or millimetres. `float` is appropriate for real-world quantities even though it has precision limits.
- **Cast complex types in steps.** Converting a `str` that looks like `"3.14"` to `int` requires two steps: `int(float("3.14"))`.
- **Be explicit in data pipelines.** If you are cleaning a CSV or parsing JSON, cast every numeric column intentionally rather than relying on implicit behaviour. Explicit casts are self-documenting.
- **🤖 AI-engineering relevance:** LLMs accept string inputs and return string outputs. Every time you call an LLM API, you cast your data to strings and parse the response. When building RAG pipelines, you cast document chunks to strings for embedding, and cast model outputs back to structured types (JSON) using `json.loads()`. Getting comfortable with `str`/`int`/`float` casting now makes those pipelines obvious instead of mysterious.

## 📌 Summary

| Function | Input types | Return type | Key behaviour |
| --- | --- | --- | --- |
| `int(x)` | `float`, `str`, `bool` | `int` | Truncates floats; parses strings with optional base |
| `float(x)` | `int`, `str`, `bool` | `float` | Adds `.0` to ints; supports scientific notation in strings |
| `str(x)` | any object | `str` | Universal string conversion; use `repr()` for debug representation |
| `bool(x)` | any object | `bool` | Falsy: `0`, `""`, `None`, empty containers; everything else is truthy |
| `round(x)` | `float`, `int` | `float` or `int` | Rounds to nearest int (or specified decimal places); does not truncate |
| `int(x, base)` | `str` | `int` | Parses string in given base (2, 8, 10, 16, etc.) |
| `.isdigit()` | string method | `bool` | Returns True if all characters are digits (positive integers only) |
| `.strip()` | string method | `str` | Removes leading and trailing whitespace |

Key takeaways:

- **Casting** is explicit type conversion using `int()`, `float()`, `str()`, or `bool()`.
- `int()` **truncates** (drops fractional part), it does not round. Use `round()` for rounding.
- Invalid casts raise `ValueError`. Guard with `try/except` or `.isdigit()` for strings.
- `bool()` follows truthiness rules: only `0`, `""`, `None`, and empty containers are falsy.
- The string `"False"` is truthy — it contains characters, even though it spells "false".
- Always `.strip()` strings from external sources (files, user input, APIs) before casting.
- Cast in **data pipelines** (CSV, JSON, API responses) is explicit and intentional, never accidental.

## 🔗 Next Lesson

Head to **[05_Strings](../05_Strings/notes.ipynb)** — Python's powerful string type, from indexing to f-strings.